# 🎯 Syndicate-Level Feature Engineering

**Phase 2 of Unit Talk ML Training Pipeline**

Creating 150+ professional-grade features to compete with syndicate groups and the best human cappers in the world.

## Feature Categories (150+ Total)
1. **Player Performance Features** (40+ features)
2. **Team Dynamics Features** (30+ features) 
3. **Market Intelligence Features** (25+ features)
4. **Temporal Features** (20+ features)
5. **Matchup Features** (20+ features)
6. **Environmental Features** (15+ features)

## Target Performance
- **Win Rate**: 55-58% (vs current 35.2% baseline)
- **ROI**: 5-8% long-term
- **CLV**: Positive closing line value
- **Sharpe Ratio**: >1.5

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Feature engineering
import sys
sys.path.append('/app/features')
from feature_engine import SyndicateFeatureEngine

print("🚀 SYNDICATE FEATURE ENGINEERING INITIALIZED")
print("=" * 50)
print(f"Target: Build 150+ features for 55-58% win rate")
print(f"Current Baseline: 35.2% (from 80,329 training samples)")
print(f"Goal: Compete with best human cappers and syndicate groups")

In [ ]:
# Load our 80,329 training samples
print("📊 Loading Unit Talk training data...")

try:
    # Load the exported training data
    data = pd.read_csv('/app/data/all_training_data.csv')
    print(f"✅ Loaded {len(data):,} training samples")
    
    # Display basic info
    print(f"\n📈 Dataset Overview:")
    print(f"  Shape: {data.shape}")
    print(f"  Date range: {data['start_time'].min()} to {data['start_time'].max()}")
    print(f"\n🏆 Sport breakdown:")
    print(data['sport'].value_counts())
    print(f"\n🎯 Results breakdown:")
    print(data['result'].value_counts())
    
except FileNotFoundError:
    print("⚠️ Training data not found. Creating sample data for development...")
    # Create sample data for development
    np.random.seed(42)
    data = pd.DataFrame({
        'prop_id': range(5000),
        'game_id': [f'game_{i//20}' for i in range(5000)],
        'sport': np.random.choice(['NBA', 'NFL', 'MLB', 'NHL'], 5000),
        'stat_type': np.random.choice(['points', 'rebounds', 'assists', 'rushing_yards'], 5000),
        'player_name': [f'Player_{i%500}' for i in range(5000)],
        'line': np.random.uniform(15, 35, 5000),
        'over_odds': np.random.randint(-120, -90, 5000),
        'under_odds': np.random.randint(-120, -90, 5000),
        'actual_value': np.random.uniform(10, 40, 5000),
        'result': np.random.choice(['won', 'lost'], 5000, p=[0.35, 0.65]),
        'start_time': pd.date_range('2024-01-01', periods=5000, freq='3H'),
        'day_of_week': np.random.randint(0, 7, 5000),
        'hour_of_day': np.random.randint(12, 23, 5000),
        'month': np.random.randint(1, 13, 5000),
        'days_until_game': np.random.randint(0, 7, 5000),
        'is_primetime': np.random.binomial(1, 0.3, 5000),
        'is_weekend': np.random.binomial(1, 0.3, 5000)
    })
    print(f"✅ Created {len(data):,} sample training records")

# Display first few rows
print(f"\n📋 Sample data:")
display(data.head())

# Check for missing values
print(f"\n🔍 Missing values:")
missing = data.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values")

In [ ]:
# Initialize the Syndicate Feature Engine
print("🔧 Initializing Syndicate Feature Engine...")

# Create feature engine
engine = SyndicateFeatureEngine(data)

# Generate all 150+ features
print("⚡ Creating syndicate-level features...")
features = engine.create_all_features()

print(f"\n✅ FEATURE ENGINEERING COMPLETE")
print(f"📊 Original features: {data.shape[1]}")
print(f"🚀 New features created: {features.shape[1] - 5}")
print(f"💪 Total feature count: {features.shape[1]}")
print(f"🎯 Ready for syndicate-level model training!")

In [ ]:
# Analyze feature distribution and importance
print("📈 FEATURE ANALYSIS")
print("=" * 30)

# Display feature categories
feature_cols = [col for col in features.columns if col not in ['prop_id', 'sport', 'stat_type', 'player_name', 'result']]

print(f"\n🔢 Feature breakdown:")
player_features = [col for col in feature_cols if col.startswith('player_')]
team_features = [col for col in feature_cols if col.startswith('team_')]
market_features = [col for col in feature_cols if any(x in col for x in ['line_', 'market_', 'closing_', 'sharp_', 'public_', 'contrarian_', 'implied_', 'true_', 'edge_'])]
temporal_features = [col for col in feature_cols if any(x in col for x in ['day_', 'hour_', 'month_', 'season_', 'playoff_', 'is_', 'days_'])]
matchup_features = [col for col in feature_cols if any(x in col for x in ['h2h_', 'position_', 'style_', 'opponent_', 'matchup_', 'exploitation_'])]
environmental_features = [col for col in feature_cols if any(x in col for x in ['game_', 'stakes_', 'pressure_', 'injury_', 'news_', 'public_attention', 'weather_', 'venue_'])]

print(f"  🏀 Player Performance: {len(player_features)} features")
print(f"  🏈 Team Dynamics: {len(team_features)} features")
print(f"  💰 Market Intelligence: {len(market_features)} features")
print(f"  ⏰ Temporal: {len(temporal_features)} features")
print(f"  🥊 Matchup: {len(matchup_features)} features")
print(f"  🌤️ Environmental: {len(environmental_features)} features")

print(f"\n📊 Sample feature values:")
display(features[feature_cols[:10]].describe())

In [ ]:
# Quick model training to test feature effectiveness
print("🤖 BASELINE MODEL TRAINING")
print("=" * 35)

# Prepare data for training
X = features[feature_cols].fillna(0)  # Handle any NaN values
y = (features['result'] == 'won').astype(int)  # Convert to binary

print(f"📊 Training data shape: {X.shape}")
print(f"🎯 Target distribution: {y.value_counts()}")
print(f"📈 Baseline win rate: {y.mean():.3f} ({y.mean()*100:.1f}%)")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train baseline Random Forest model
print("\n🌲 Training Random Forest baseline...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Evaluate performance
train_acc = rf_model.score(X_train, y_train)
test_acc = rf_model.score(X_test, y_test)
y_pred = rf_model.predict(X_test)

print(f"\n📊 BASELINE RESULTS:")
print(f"  Training accuracy: {train_acc:.3f} ({train_acc*100:.1f}%)")
print(f"  Testing accuracy: {test_acc:.3f} ({test_acc*100:.1f}%)")
print(f"  Target improvement: {55 - test_acc*100:.1f}% points to reach 55% win rate")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🔥 Top 10 most important features:")
for i, (_, row) in enumerate(feature_importance.head(10).iterrows()):
    print(f"  {i+1:2d}. {row['feature']:25s} ({row['importance']:.4f})")

In [ ]:
# Visualize feature importance and performance
print("📊 FEATURE VISUALIZATION")
print("=" * 30)

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Feature importance (top 15)
top_features = feature_importance.head(15)
axes[0,0].barh(top_features['feature'], top_features['importance'])
axes[0,0].set_title('🔥 Top 15 Feature Importance')
axes[0,0].set_xlabel('Importance Score')

# 2. Win rate by sport
sport_performance = features.groupby('sport')['result'].apply(lambda x: (x == 'won').mean())
axes[0,1].bar(sport_performance.index, sport_performance.values)
axes[0,1].set_title('🏆 Win Rate by Sport')
axes[0,1].set_ylabel('Win Rate')
axes[0,1].axhline(y=0.55, color='red', linestyle='--', label='Target (55%)')
axes[0,1].legend()

# 3. Feature correlation heatmap (sample)
sample_features = feature_cols[:10]  # First 10 features
corr_matrix = X[sample_features].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1,0])
axes[1,0].set_title('🔗 Feature Correlation (Sample)')

# 4. Performance distribution
axes[1,1].hist([y_train, y_test], label=['Train', 'Test'], alpha=0.7, bins=2)
axes[1,1].set_title('🎯 Win/Loss Distribution')
axes[1,1].set_xlabel('Result (0=Loss, 1=Win)')
axes[1,1].set_ylabel('Count')
axes[1,1].legend()

plt.tight_layout()
plt.show()

print(f"\n✅ Visualizations complete")
print(f"🚀 Ready for Phase 3: Base Model Training")

In [ ]:
# Save engineered features for model training
print("💾 SAVING FEATURES FOR MODEL TRAINING")
print("=" * 40)

# Save complete feature set
features.to_csv('/app/data/syndicate_features_complete.csv', index=False)
print(f"✅ Complete features saved: {features.shape}")

# Save feature metadata
feature_metadata = {
    'total_features': len(feature_cols),
    'player_features': len(player_features),
    'team_features': len(team_features),
    'market_features': len(market_features),
    'temporal_features': len(temporal_features),
    'matchup_features': len(matchup_features),
    'environmental_features': len(environmental_features),
    'baseline_accuracy': test_acc,
    'target_improvement': 55 - test_acc*100,
    'top_features': feature_importance.head(20).to_dict('records'),
    'sports_coverage': features['sport'].value_counts().to_dict(),
    'created_at': datetime.now().isoformat()
}

import json
with open('/app/data/feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print(f"✅ Feature metadata saved")
print(f"\n🎯 PHASE 2 COMPLETE: FEATURE ENGINEERING")
print(f"📈 Features created: {len(feature_cols)}")
print(f"🎪 Baseline model: {test_acc*100:.1f}% accuracy")
print(f"🚀 Ready for Phase 3: Training 7 sport-specific models")
print(f"🏆 Target: 55-58% win rate to rival syndicate groups")

## 🎯 Next Steps: Phase 3 Model Training

**Phase 2 Complete**: 150+ syndicate-level features engineered

**Next Phase**: Train 7 sport-specific ML models
- NBA model (XGBoost + Neural Network)
- NFL model (Specialized for football dynamics)
- MLB model (Seasonal and pitcher-specific features)
- NHL model (Hockey-specific momentum)
- NCAAF model (College football patterns)
- NCAAB model (Tournament dynamics)
- WNBA model (Women's basketball specifics)

**Target Performance**:
- Win Rate: 55-58% (current baseline ~35%)
- ROI: 5-8% long-term
- CLV: Positive closing line value
- Sharpe Ratio: >1.5

**Ready to compete with the best human cappers and syndicate groups in the world! 🚀**